In [2]:
import wandb
import pandas as pd
import numpy as np

import json

api = wandb.Api()

In [4]:
def linearize_dict(d, parent_key='', sep='.'):
    """
    Linearizes a nested dictionary into a flat dictionary with keys in the format `key1.key2`.

    :param d: The dictionary to linearize.
    :param parent_key: The current key being processed (used for recursion).
    :param sep: The separator between keys.
    :return: A flattened dictionary.
    """
    items = []
    
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(linearize_dict(v, new_key, sep).items())
        else:
            items.append((new_key, v))
    
    return dict(items)

def get_runs_df(runs):
    runs_list = []
    for run in runs: 
        runs_list.append({
            **run.summary._json_dict,
            **linearize_dict({k: v for k,v in run.config.items()
            if not k.startswith('_')}),
            **linearize_dict(run.metadata if run.metadata else {}),
            **{"name": run.name, "id": run.id}
            })

    runs_df = pd.DataFrame(runs_list)
    return runs_df

In [58]:
iauc_dauc_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.n_steps": 75},
            {"tags": {"$nin": ["Broken"]}},
            {"$or": [
                {'config.dataset.datasets.val_pascal5i_N1K5.name': "pascal"},
                {'config.dataset.datasets.val_coco20i_N1K5.name': "coco"},
            ]}
        ]
    }
)

In [81]:
iauc_dauc_runs_df = get_runs_df(iauc_dauc_runs)

In [82]:
res_df = iauc_dauc_runs_df.copy()

dataset_names = ["val_pascal5i_N1K5", "val_coco20i_N1K5"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1)

iauc_cols = [f"{name}_iauc" for name in dataset_names]
dauc_cols = [f"{name}_dauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."
assert res_df[dauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."


renamings_dict = {
    "explainer.name": "Explanation Method",
    "config.metric.n_steps": "N. Steps",
    "config.metric.iauc": "IAUC",
    "config.metric.dauc": "DAUC",
    "model": "Model",
}
res_df = res_df.rename(columns=renamings_dict)

value_renamings_dict = {
    "integrated_gradients": "Integrated Gradients",
    "saliency": "Saliency",
    "affinity": "Affinity Explainer (ours)",
    "random": "Random",
    "pascal": "Pascal 5^i",
    "coco": "COCO 20^i",
    "dcama": "DCAMA",
    "dmtnet": "DMTNet",
}
res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["IAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["DAUC"] = res_df[dauc_cols].fillna(0).sum(axis=1)

res_df["Diff."] = res_df["IAUC"] - res_df["DAUC"]

cols = ["Model", "Explanation Method", "Dataset"]
res_df = res_df[cols + ["IAUC", "DAUC", "Diff."]]

# Define custom sort order
dataset_order = {"COCO 20^i": 0, "Pascal 5^i": 1}
metric_order = {'IAUC': 0, 'DAUC': 1, 'Diff.': 2}
method_order = {
    'Random': 0,
    'Integrated Gradients': 1,
    'Saliency': 2,
    'Affinity Explainer': 3,
}
model_order = {
    'DCAMA': 0,
    'DMTNet': 1,
}

res_df = res_df.pivot_table(
    index=['Model', 'Explanation Method'],
    columns='Dataset',
    values=['IAUC', 'DAUC', 'Diff.']
).swaplevel(axis=1).sort_index(axis=1)


new_cols  = sorted(res_df.columns, key=lambda x: (dataset_order[x[0]], metric_order[x[1]]))
res_df = res_df[new_cols]
# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Model', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(model_order) if x.name == 'Model' else x.map(method_order)
)

iauc_dauc_df = (res_df*100).round(2)

iauc_dauc_df

Dataset                          COCO 20^i               Pascal 5^i         \
                                      IAUC   DAUC  Diff.       IAUC   DAUC   
Model  Explanation Method                                                    
DCAMA  Random                        73.97  74.16  -0.19      83.14  83.35   
       Integrated Gradients          68.46  79.27 -10.81      79.53  86.19   
       Saliency                      77.35  65.54  11.81      83.90  78.31   
       Affinity Explainer (ours)     80.58  54.84  25.73      87.16  62.25   
DMTNet Random                        46.08  46.03   0.05      46.79  46.85   
       Integrated Gradients          44.59  52.60  -8.02      48.02  55.16   
       Saliency                      62.59  66.74  -4.16      65.96  68.59   
       Affinity Explainer (ours)     69.65  65.38   4.27      75.03  66.33   

Dataset                                  
                                  Diff.  
Model  Explanation Method                
DCAMA  Random                     -0.21  
       Integrated Gradients       -6.67  
       Saliency                    5.60  
       Affinity Explainer (ours)  24.92  
DMTNet Random                     -0.06  
       Integrated Gradients       -7.14  
       Saliency                   -2.63  
       Affinity Explainer (ours)   8.70

In [29]:
from tabulate import tabulate

latex_tab = tabulate(res_df, headers='keys', tablefmt='latex_booktabs', showindex=True)
print(latex_tab)

\begin{tabular}{lrrrrrr}
\toprule
                                         &   ('COCO 20\^{}i', 'iAUC') &   ('COCO 20\^{}i', 'dAUC') &   ('COCO 20\^{}i', 'Diff.') &   ('Pascal 5\^{}i', 'iAUC') &   ('Pascal 5\^{}i', 'dAUC') &   ('Pascal 5\^{}i', 'Diff.') \\
\midrule
 ('DCAMA', 'Random')                     &                   73.97 &                   74.16 &                    -0.19 &                    83.14 &                    83.35 &                     -0.21 \\
 ('DCAMA', 'Integrated Gradients')       &                   68.46 &                   79.27 &                   -10.81 &                    79.53 &                    86.19 &                     -6.67 \\
 ('DCAMA', 'Saliency')                   &                   77.35 &                   65.54 &                    11.81 &                    83.9  &                    78.31 &                      5.6  \\
 ('DCAMA', 'Affinity Explainer (ours)')  &                   80.58 &                   54.84 &                    25.73

## IAUC %

In [61]:
iauc_p_runs = api.runs(
    "pasqualedem/affex",
    filters={
        "$and": [
            {"config.metric.percentage": {"$ne": None}},
            {"tags": {"$nin": ["Broken"]}},
            {"$or": [
                {'config.dataset.datasets.val_pascal5i_N1K5.name': "pascal"},
                {'config.dataset.datasets.val_coco20i_N1K5.name': "coco"},
            ]}
        ]
    }
)

In [62]:
iauc_p_runs_df = get_runs_df(iauc_p_runs)

In [83]:
res_df = iauc_p_runs_df.copy()

dataset_names = ["val_pascal5i_N1K5", "val_coco20i_N1K5"]

dataset_cols = [f'dataset.datasets.{name}.name' for name in dataset_names]
res_df["Dataset"] = res_df[dataset_cols].fillna("").sum(axis=1)

iauc_cols = [f"{name}_iauc" for name in dataset_names]
assert res_df[iauc_cols].isna().any(axis=1).all(), "More than one dataset is present in the same run, please check the dataset columns."


renamings_dict = {
    "explainer.name": "Explanation Method",
    "config.metric.n_steps": "N. Steps",
    "config.metric.iauc": "IAUC",
    "config.metric.dauc": "DAUC",
    "model": "Model",
    "metric.percentage": "Percentage",
    "metric.loss": "Loss",
}
res_df = res_df.rename(columns=renamings_dict)

value_renamings_dict = {
    "integrated_gradients": "Integrated Gradients",
    "saliency": "Saliency",
    "affinity": "Affinity Explainer (ours)",
    "random": "Random",
    "pascal": "Pascal 5^i",
    "coco": "COCO 20^i",
    "dcama": "DCAMA",
    "dmtnet": "DMTNet",
}
res_df["Explanation Method"] = res_df["Explanation Method"].replace(value_renamings_dict)
res_df["Dataset"] = res_df["Dataset"].replace(value_renamings_dict)
res_df["Model"] = res_df["Model"].replace(value_renamings_dict)


res_df["iAUC"] = res_df[iauc_cols].fillna(0).sum(axis=1)
res_df["Loss"] = res_df["Loss"].fillna(False)

cols = ["Model", "Explanation Method", "Dataset", "Percentage", "Loss"]
res_df = res_df[cols + ["iAUC"]]

# Define custom sort order
dataset_order = {"COCO 20^i": 0, "Pascal 5^i": 1}
method_order = {
    'Random': 0,
    'Integrated Gradients': 1,
    'Saliency': 2,
    'Affinity Explainer': 3,
}
model_order = {
    'DCAMA': 0,
    'DMTNet': 1,
}

res_df = res_df.pivot_table(
    index=['Model', 'Explanation Method'],
    columns=['Dataset', 'Percentage', "Loss"],
    values='iAUC'
).sort_index(axis=1)

perc_map = {
    False: "IAUC@",
    True: "mIoULoss@"
}

# Flatten the MultiIndex columns
# Collapse last two levels into one
res_df.columns = pd.MultiIndex.from_tuples([
    (lvl0, f"{perc_map[lvl2]}{lvl1}") for lvl0, lvl1, lvl2 in res_df.columns
])


# Sort by Model and then by Explanation Method using the provided orderings
res_df = res_df.sort_values(
    by=['Model', 'Explanation Method'],  # this ensures level names exist and are used
    key=lambda x: x.map(model_order) if x.name == 'Model' else x.map(method_order)
)

iauc_p_df = (res_df*100).round(2)

iauc_p_df

/tmp/ipykernel_3789150/692976512.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  res_df["Loss"] = res_df["Loss"].fillna(False)


COCO 20^i                              \
                                 IAUC@0.01 mIoULoss@0.01 mIoULoss@0.05   
Model  Explanation Method                                                
DCAMA  Random                        43.36         41.42         31.62   
       Integrated Gradients          45.34         34.70         24.04   
       Saliency                      44.87         37.61         23.60   
       Affinity Explainer (ours)     60.82         17.41          8.85   
DMTNet Random                        60.36         24.93         31.88   
       Integrated Gradients          61.45         22.77         23.40   
       Saliency                      62.58         20.80         17.90   
       Affinity Explainer (ours)     67.30         10.53          4.49   

                                 Pascal 5^i                              
                                  IAUC@0.01 mIoULoss@0.01 mIoULoss@0.05  
Model  Explanation Method                                                
DCAMA  Random                         41.71         55.24         33.42  
       Integrated Gradients           43.34         44.73         24.73  
       Saliency                       44.22         46.48         24.56  
       Affinity Explainer (ours)      66.82         13.78          5.72  
DMTNet Random                         60.35         33.89         46.01  
       Integrated Gradients           63.67         31.12         35.51  
       Saliency                       65.11         28.91         26.49  
       Affinity Explainer (ours)      66.80         13.34          5.32

---

In [91]:
# joining iauc_dauc_df and iauc_p_df on index
final = iauc_dauc_df.join(iauc_p_df)

metric_order = {
    'IAUC': 0,
    'DAUC': 1,
    'Diff.': 2,
    'IAUC@0.01': 3,
    'IAUC@0.05': 4,
    'mIoULoss@0.01': 5,
    'mIoULoss@0.05': 6,
}

new_cols  = sorted(final.columns, key=lambda x: (dataset_order[x[0]], metric_order[x[1]]))
final = final[new_cols]
final

COCO 20^i                          \
                                      IAUC   DAUC  Diff. IAUC@0.01   
Model  Explanation Method                                            
DCAMA  Random                        73.97  74.16  -0.19     43.36   
       Integrated Gradients          68.46  79.27 -10.81     45.34   
       Saliency                      77.35  65.54  11.81     44.87   
       Affinity Explainer (ours)     80.58  54.84  25.73     60.82   
DMTNet Random                        46.08  46.03   0.05     60.36   
       Integrated Gradients          44.59  52.60  -8.02     61.45   
       Saliency                      62.59  66.74  -4.16     62.58   
       Affinity Explainer (ours)     69.65  65.38   4.27     67.30   

                                                             Pascal 5^i  \
                                 mIoULoss@0.01 mIoULoss@0.05       IAUC   
Model  Explanation Method                                                 
DCAMA  Random                            41.42         31.62      83.14   
       Integrated Gradients              34.70         24.04      79.53   
       Saliency                          37.61         23.60      83.90   
       Affinity Explainer (ours)         17.41          8.85      87.16   
DMTNet Random                            24.93         31.88      46.79   
       Integrated Gradients              22.77         23.40      48.02   
       Saliency                          20.80         17.90      65.96   
       Affinity Explainer (ours)         10.53          4.49      75.03   

                                                                        \
                                   DAUC  Diff. IAUC@0.01 mIoULoss@0.01   
Model  Explanation Method                                                
DCAMA  Random                     83.35  -0.21     41.71         55.24   
       Integrated Gradients       86.19  -6.67     43.34         44.73   
       Saliency                   78.31   5.60     44.22         46.48   
       Affinity Explainer (ours)  62.25  24.92     66.82         13.78   
DMTNet Random                     46.85  -0.06     60.35         33.89   
       Integrated Gradients       55.16  -7.14     63.67         31.12   
       Saliency                   68.59  -2.63     65.11         28.91   
       Affinity Explainer (ours)  66.33   8.70     66.80         13.34   

                                                
                                 mIoULoss@0.05  
Model  Explanation Method                       
DCAMA  Random                            33.42  
       Integrated Gradients              24.73  
       Saliency                          24.56  
       Affinity Explainer (ours)          5.72  
DMTNet Random                            46.01  
       Integrated Gradients              35.51  
       Saliency                          26.49  
       Affinity Explainer (ours)          5.32

In [92]:
# turn the two level index into two columns
final_to_latex = final.reset_index()

latex_tab = tabulate(final_to_latex, headers='keys', tablefmt='latex_booktabs', showindex=False)
print(latex_tab)

\begin{tabular}{llrrrrrrrrrrrr}
\toprule
 ('Model', '')   & ('Explanation Method', '')   &   ('COCO 20\^{}i', 'IAUC') &   ('COCO 20\^{}i', 'DAUC') &   ('COCO 20\^{}i', 'Diff.') &   ('COCO 20\^{}i', 'IAUC@0.01') &   ('COCO 20\^{}i', 'mIoULoss@0.01') &   ('COCO 20\^{}i', 'mIoULoss@0.05') &   ('Pascal 5\^{}i', 'IAUC') &   ('Pascal 5\^{}i', 'DAUC') &   ('Pascal 5\^{}i', 'Diff.') &   ('Pascal 5\^{}i', 'IAUC@0.01') &   ('Pascal 5\^{}i', 'mIoULoss@0.01') &   ('Pascal 5\^{}i', 'mIoULoss@0.05') \\
\midrule
 DCAMA           & Random                       &                   73.97 &                   74.16 &                    -0.19 &                        43.36 &                            41.42 &                            31.62 &                    83.14 &                    83.35 &                     -0.21 &                         41.71 &                             55.24 &                             33.42 \\
 DCAMA           & Integrated Gradients         &                   68.46 &     